# Huấn luyện — Lê Văn Quân
Random Forest và Extra Trees.

Chạy sau `chuanbi_dulieu.ipynb`. Giữ đủ 12 đặc trưng, dùng chung Pipeline và 5 fold CV. Notebook này chỉ đọc train, không đánh giá test.


In [1]:
from pathlib import Path
import sys
import pandas as pd
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / 'data' / 'winequality-red.csv').is_file()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Đặt notebook trong thư mục notebooks của dự án.')
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
OUT = PROJECT_ROOT / 'output'


## 1. Cấu hình mô hình và miền siêu tham số
Sửa cấu hình chung tại `src/pipelines.py` trước khi cả hai chạy. Giữ cùng `n_iter=8`.

In [3]:
import pandas as pd
from IPython.display import display

from src.pipelines import model_pipelines, search_spaces
from src.split_experiment import GROUPS, run_group

OWNER = "quan"
models = model_pipelines()
spaces = search_spaces()

# 1. Bảng cấu hình Pipeline
pipeline_rows = []

for name in GROUPS[OWNER]:
    steps = models[name].named_steps

    pipeline_rows.append({
        "Mô hình": name,
        "Điền khuyết": "Trung vị",
        "Chuẩn hóa": "Có" if "scaler" in steps else "Không",
        "Tìm tham số": "Có" if name in spaces else "Không"
    })

print("CẤU HÌNH PIPELINE")
display(pd.DataFrame(pipeline_rows).style.hide(axis="index"))

# 2. Bảng miền tìm kiếm siêu tham số
param_rows = []

for name in GROUPS[OWNER]:
    for param, values in spaces.get(name, {}).items():
        param_rows.append({
            "Mô hình": name,
            "Tham số": param.removeprefix("model__"),
            "Giá trị thử": ", ".join(map(str, values))
        })

print("\nMIỀN TÌM KIẾM SIÊU THAM SỐ")
if param_rows:
    display(pd.DataFrame(param_rows).style.hide(axis="index"))

CẤU HÌNH PIPELINE


Mô hình,Điền khuyết,Chuẩn hóa,Tìm tham số
Random Forest,Trung vị,Không,Có
Extra Trees,Trung vị,Không,Có



MIỀN TÌM KIẾM SIÊU THAM SỐ


Mô hình,Tham số,Giá trị thử
Random Forest,n_estimators,"100, 200"
Random Forest,max_depth,"None, 10, 20"
Random Forest,min_samples_leaf,"1, 3"
Random Forest,max_features,"sqrt, 1.0"
Random Forest,class_weight,"None, balanced"
Extra Trees,n_estimators,"100, 200"
Extra Trees,max_depth,"None, 10, 20"
Extra Trees,min_samples_leaf,"1, 3"
Extra Trees,max_features,"sqrt, 1.0"
Extra Trees,class_weight,"None, balanced"


## 2. CV cấu hình ban đầu và tìm kiếm siêu tham số
Tiền xử lý được học trong mỗi fold; F1 macro là chỉ số chọn. Lưu điểm train/validation, trung bình và độ lệch chuẩn mẫu.

In [4]:
initial, tuned = run_group(PROJECT_ROOT, OWNER, n_iter=8, n_jobs=2)
display(initial)
display(tuned)

CV: quan Random Forest
CV: quan Extra Trees


,model,train_accuracy_mean,train_accuracy_std,val_accuracy_mean,val_accuracy_std,train_balanced_accuracy_mean,train_balanced_accuracy_std,val_balanced_accuracy_mean,val_balanced_accuracy_std,train_f1_macro_mean,train_f1_macro_std,val_f1_macro_mean,val_f1_macro_std,train_f1_weighted_mean,train_f1_weighted_std,val_f1_weighted_mean,val_f1_weighted_std,fit_time_mean
0,Random Forest,1.0,0.0,0.631584,0.016732,1.0,0.0,0.596968,0.015718,1.0,0.0,0.609857,0.016808,1.0,0.0,0.628416,0.016291,1.477577
1,Extra Trees,1.0,0.0,0.638163,0.014900,1.0,0.0,0.601606,0.015508,1.0,0.0,0.613950,0.017353,1.0,0.0,0.634202,0.015201,0.893237


,model,stage,f1_macro,std,train_f1_macro,best_params
0,Random Forest,tuned,0.612789,0.014874,0.958666,"{'model__n_estimators': 200, 'model__min_sampl..."
1,Extra Trees,tuned,0.613218,0.013822,1.000000,"{'model__n_estimators': 100, 'model__min_sampl..."
